In [1]:
import pandas as pd
from collections import defaultdict

from golden_data.config import get_raw_data_path, DATA_INTERIM
from golden_data.custom_field_scan import parse_cf_cell
from golden_data.normalization import load_mapping, normalize_series_with_mapping

In [2]:
raw_path = get_raw_data_path()
df = pd.read_csv(raw_path, low_memory=False)

# Choose an identifier column
id_col_candidates = ["Product ID", "ID", "SKU", "Product Code/SKU"]
id_col = None
for c in id_col_candidates:
    if c in df.columns:
        id_col = c
        break

if id_col is None:
    # Fallback: use index as surrogate ID
    df["RowID"] = df.index
    id_col = "RowID"

id_col

'ID'

In [3]:
rows = []
for _, row in df.iterrows():
    pid = row[id_col]
    for name, value in parse_cf_cell(row.get("Custom Fields")):
        if name in ("Material", "Finish") and value:
            rows.append(
                {
                    id_col: pid,
                    "Field_Name": name,
                    "Raw_Value": str(value),
                }
            )

attr_df = pd.DataFrame(rows)
attr_df.head()

,ID,Field_Name,Raw_Value
0,3375,Material,Soft Urethane
1,3376,Material,Soft Urethane
2,3378,Material,Soft Urethane
3,3379,Material,Soft Urethane
4,3380,Material,Soft Urethane


In [4]:
attr_pivot = (
    attr_df
    .pivot_table(index=id_col, columns="Field_Name", values="Raw_Value", aggfunc="first")
    .reset_index()
)

attr_pivot.head()

Field_Name,ID,Finish,Material
0,3375,NaN,Soft Urethane
1,3376,NaN,Soft Urethane
2,3378,NaN,Soft Urethane
3,3379,NaN,Soft Urethane
4,3380,NaN,Soft Urethane


In [5]:
from golden_data.normalization import get_mapping_path

material_mapping = load_mapping("Material", normalized_column="normalized_material")
finish_mapping = load_mapping("Finish", normalized_column="normalized_finish")

material_mapping.head(), finish_mapping.head()

(                    raw_value         normalized_material  notes
 0                       Steel                       Steel    NaN
 1  300 SERIES STAINLESS STEEL  300 SERIES STAINLESS STEEL    NaN
 2             Stainless Steel             Stainless Steel    NaN
 3                    Aluminum                    Aluminum    NaN
 4                   Nylon 6/6                   Nylon 6/6    NaN,
                                 raw_value  \
 0  Passivated and/or tested per ASTM A380   
 1                                 Natural   
 2             Zinc Plate, Bright chromate   
 3                             Powder Coat   
 4                              Passivated   
 
                         normalized_finish  notes  
 0  Passivated and/or tested per ASTM A380    NaN  
 1                                 Natural    NaN  
 2             Zinc Plate, Bright chromate    NaN  
 3                             Powder Coat    NaN  
 4                              Passivated    NaN  )

In [6]:
attr_pivot["Material_Normalized"] = normalize_series_with_mapping(
    attr_pivot["Material"],
    material_mapping,
    normalized_column="normalized_material",
    policy="flag",   # flag unmapped values so you can see them
)

attr_pivot["Finish_Normalized"] = normalize_series_with_mapping(
    attr_pivot["Finish"],
    finish_mapping,
    normalized_column="normalized_finish",
    policy="flag",
)

attr_pivot.head(20)

Field_Name,ID,Finish,Material,Material_Normalized,Finish_Normalized
0,3375,NaN,Soft Urethane,Soft Urethane,NaN
1,3376,NaN,Soft Urethane,Soft Urethane,NaN
2,3378,NaN,Soft Urethane,Soft Urethane,NaN
3,3379,NaN,Soft Urethane,Soft Urethane,NaN
4,3380,NaN,Soft Urethane,Soft Urethane,NaN
5,3381,NaN,Soft Urethane,Soft Urethane,NaN
6,3382,NaN,Soft Urethane,Soft Urethane,NaN
7,3383,NaN,Soft Urethane,Soft Urethane,NaN
8,3384,NaN,Soft Urethane,Soft Urethane,NaN
9,3385,NaN,Soft Urethane,Soft Urethane,NaN


In [7]:
attr_pivot["Material_Normalized"] = normalize_series_with_mapping(
    attr_pivot["Material"],
    material_mapping,
    normalized_column="normalized_material",
    policy="flag",   # flag unmapped values so you can see them
)

attr_pivot["Finish_Normalized"] = normalize_series_with_mapping(
    attr_pivot["Finish"],
    finish_mapping,
    normalized_column="normalized_finish",
    policy="flag",
)

attr_pivot.head(20)

Field_Name,ID,Finish,Material,Material_Normalized,Finish_Normalized
0,3375,NaN,Soft Urethane,Soft Urethane,NaN
1,3376,NaN,Soft Urethane,Soft Urethane,NaN
2,3378,NaN,Soft Urethane,Soft Urethane,NaN
3,3379,NaN,Soft Urethane,Soft Urethane,NaN
4,3380,NaN,Soft Urethane,Soft Urethane,NaN
5,3381,NaN,Soft Urethane,Soft Urethane,NaN
6,3382,NaN,Soft Urethane,Soft Urethane,NaN
7,3383,NaN,Soft Urethane,Soft Urethane,NaN
8,3384,NaN,Soft Urethane,Soft Urethane,NaN
9,3385,NaN,Soft Urethane,Soft Urethane,NaN


In [9]:
from golden_data.data_profiling import profile_raw_data
from golden_data.custom_field_scan import scan_custom_fields
from golden_data.config import DATA_INTERIM
from collections import Counter

import pandas as pd

# 1. Load raw data (optional, but you already do this)
df = profile_raw_data()  # returns full raw DataFrame

# 2. Custom field summary
cf_summary = scan_custom_fields(max_rows=5000)
print("cf_summary rows:", len(cf_summary))

# 3. Phase-1 coverage + value counts (from earlier)
phase1_fields = [
    "Material",
    "Finish",
    "Type",
    "Thread",
    "Length",
    "Diameter",
    "Width",
    "Height",
    "Head Style",
    "RoHS Compliant",
]

from golden_data.config import get_raw_data_path
from golden_data.custom_field_scan import parse_cf_cell

raw_path = get_raw_data_path()
df_raw = pd.read_csv(raw_path, low_memory=False)

if "Item" in df_raw.columns:
    products = df_raw[df_raw["Item"] == "Product"].copy()
else:
    products = df_raw.copy()

total_products = len(products)

field_value_counts = {name: Counter() for name in phase1_fields}
product_has_field = {name: 0 for name in phase1_fields}

for _, row in products.iterrows():
    for name, value in parse_cf_cell(row.get("Custom Fields")):
        if name in phase1_fields and value:
            field_value_counts[name][value] += 1
            product_has_field[name] += 1

coverage_rows = []
for name in phase1_fields:
    cnt = product_has_field[name]
    pct = 100.0 * cnt / total_products if total_products else 0.0
    unique_vals = len(field_value_counts[name])
    coverage_rows.append(
        {
            "Field_Name": name,
            "Products_With_Value": cnt,
            "Coverage_Percent": round(pct, 2),
            "Unique_Values": unique_vals,
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values("Coverage_Percent", ascending=False)

# 4. Top values for Material / Finish
def show_top_values(field_name, top_n=50):
    cnt = field_value_counts[field_name]
    top = cnt.most_common(top_n)
    return pd.DataFrame(top, columns=[f"{field_name}_Value", "Count"])

material_top = show_top_values("Material", top_n=50)
finish_top = show_top_values("Finish", top_n=50)

Profiling raw dataset: /Users/csmiller/projects/bossard-golden-data/data/raw/bossard_raw.csv

Rows: 175917, Columns: 50

                                                 Raw Data Overview                                                 
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric        ┃ Value                                                                                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Rows          │ 175917                                                                                          │
│ Columns       │ 50                                                                                              │
│ First columns │ Item, ID, Name, Type, SKU, Options, Inventory Tracking, Current Stock, Low Stock, Price, Cost   │
│               │ Price, Retail Price, Sale Price, Brand ID, Channels ...                                         │
└───────────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

cf_summary rows: 63


In [10]:
from golden_data.scripts.demo_scan import cf_summary
from golden_data.markdown_utils import df_to_markdown
from golden_data.ai_taxonomist import analyze_custom_fields, summarize_attribute_strategy

# Assume you already have:
# cf_summary, coverage_df, material_top, finish_top

cf_md = df_to_markdown(cf_summary.head(25))
coverage_md = df_to_markdown(coverage_df)
material_md = df_to_markdown(material_top.head(20))
finish_md = df_to_markdown(finish_top.head(20))

detailed_analysis = analyze_custom_fields(cf_md)

strategy_summary = summarize_attribute_strategy(
    cf_md=cf_md,
    coverage_md=coverage_md,
    material_md=material_md,
    finish_md=finish_md,
    mapping_status_notes="""
- Material and Finish top-value lists exported from Notebook 02.
- Initial mapping CSVs created in /mappings for review.
- Normalization helpers in normalization.py ready for Phase 1 pilot.
""",
)

print(detailed_analysis)
print("\n\n================ STRATEGY SUMMARY ================\n\n")
print(strategy_summary)

   Field_Name  Product_Count  Usage_Percent  \
0    Material           2479          49.58   
1  Spec Sheet           1975          39.50   
2        Type           1506          30.12   
3      Length           1437          28.74   
4       Color           1297          25.94   

                                      Example_Values  
0  Steel, Soft Urethane, Nylon 6/6, Polypropylene...  
1  <a href=/content/ProductSpecSheets/FCIAMPE0000...  
2  Single Row, Vertical, Flexible, Light Weight, ...  
3         60 in., 8.000  in., 11 in., 12 in., 75 in.  
4                Black, Natural, Bright, White, Gray  
### Narrative Overview

The provided BigCommerce custom field export reveals a moderately inconsistent and partially sparse product attribute dataset, posing challenges for Bossard’s 2026 marketing objectives. Several fundamental issues undermine taxonomy standardization, faceted navigation, and AI-driven search capabilities—critical for digital excellence and vertical-specific demand